In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
time: 2.04 s (started: 2023-04-26 09:53:39 +00:00)


In [1]:
!pip3 install --upgrade ipython-autotime
%load_ext autotime
!pip3 install sentence_transformers=='2.2.2'

!pip3 install emoji=='2.2.0'

import pandas as pd
import numpy as np
# import pickle5 as pickle
import re
from nltk.stem import WordNetLemmatizer
import nltk
from tqdm.notebook import tqdm
tqdm.pandas()
import requests

from termcolor import colored
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

nb_workers=10

import os
from datetime import datetime
import pickle
import ast
import re
import emoji
from sentence_transformers import SentenceTransformer

lemmatizer = WordNetLemmatizer()
# punctuations = """!"$%()*+,*/:#»«'";“<=>?[\]^`”{.}~"""
punctuations = """!"$%()*+,*/:»«";“<=>?[\]^`”|{.}~"""
def clean_text_local(row):
#     row = re.sub(r'[^\x00-\x7F]+', r' \g<0> ',row)
    row = ''.join(' ' + char if emoji.is_emoji(char) else char for char in row)
    row = emoji.demojize(row)
    
#     row = re.sub('\+|>','',row).replace('<','\\').encode().decode('unicode-escape')
#     row=re.sub('\x0c',' ',row)
#     row=re.sub('\u200c',' ',row)
    # row=re.sub('\n\n',' . ',row)
    # row=re.sub('\\n',' ',row)
#     row=re.sub('\\\\n',' ',row)

    row=re.sub('\\\\n',' ',row)
#     row=re.sub('\n',' ',row)
    
    row=re.sub("-?NEWLINE_TOKEN", " ",row)
    row=re.sub("TAB_TOKEN", " ",row)
    row=re.sub("Alternate option=", "",row)
    row=re.sub('RT',' ',row)
    
    row=re.sub("@[A-Za-z0-9_]+"," ",row)
#     row=re.sub("#[A-Za-z0-9_]+"," ",row)
    row=re.sub("http\S+|www.\S+"," ",row)
    row=re.sub("\["," ",row)
    row=re.sub("\]"," ",row)
    row=re.sub("\("," ",row)
    row=re.sub("\)"," ",row)
    row=re.sub("^[A-Za-z]+: ","",row)
    # row = row.lower()

#     row = re.sub(r'<.*?>', '', row)
    row = re.sub("&amp","&",row)
    
#     row=re.sub("\d","",row)
#     row=re.sub(r"\b\d+\b|\b(?![i])[a-z]\b","",row) # removes every isolated number or char except i
    # row = ' '.join([lemmatizer.lemmatize(w) for w in nltk.word_tokenize(row)])
    # row=re.sub('-',' ',row)
    # row=re.sub('_',' ',row)
    # row=re.sub('nan','',row)
#     row=re.sub(':',' ',row)
#     no_punct = ""
#     for char in row:
#         if char not in punctuations:
#             no_punct = no_punct + char
    row = row.strip()
    row = re.sub('\s+',' ',row)
    return row
    # return row

# load intents dataset
# data_sample = pd.read_csv('../data/processed/data_sample.csv')


# define the document embedding models to use for comparison
# module_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
# model_use = hub.load(module_url)
# print('loaded USE', flush=True)
model_st1 = SentenceTransformer('all-mpnet-base-v2') # version ='1'
print('loaded all-mpnet-base-v2', flush=True)
# model_st2 = SentenceTransformer('all-MiniLM-L6-v2')
# print('loaded all-MiniLM-L6-v2', flush=True)
# model_st3 = SentenceTransformer('paraphrase-mpnet-base-v2')
# print('loaded paraphrase-mpnet-base-v2', flush=True)

def embed(model, model_type, sentences):
    """
    wrapper function for generating message embeddings
    """
    if model_type == 'use':
        embeddings = model(list(sentences))
    elif model_type == 'sentence transformer':
        embeddings = model.encode(list(sentences),show_progress_bar=True)

    return embeddings

def split_embed(model_, model_type_, all_sentences_, n_partitions_=20):
    all_sentences_ = np.array_split(all_sentences_, indices_or_sections = n_partitions_)
    for i, new_sentences in enumerate(all_sentences_):
        if i==0:
            # print(colored('\nbeginning section {}'.format(i+1),'yellow'),flush=True)
            all_embeddings = embed(model_, model_type_, new_sentences)
        if i>0:
            # print(colored('\nbeginning section {}'.format(i+1),'yellow'),flush=True)
            new_embeddings = embed(model_, model_type_, new_sentences)
            all_embeddings = np.concatenate((all_embeddings,new_embeddings),axis=0)
    return all_embeddings

def vector_to_column(df,vectors,column_name):
    df[column_name]=0
    df[column_name]=df[column_name].astype(object)
    df.reset_index(drop=True,inplace=True)
    indices=df.index
    for i in indices:
        df.at[i, column_name]=vectors[i]
    return df

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
loaded all-mpnet-base-v2
time: 18.4 s (started: 2023-04-26 09:54:11 +00:00)


In [2]:
import pandas as pd
data = pd.read_hdf('/content/drive/MyDrive/Kialo vs ChatBots/kialo_popular_tabular_format.hdf')
data

,text_unclean,text,text_index,stance,discussion_id,discussion_name,file_name,stance_to_root,distance_to_root,text_vectorized,all-mpnet-base-v2,language,tags_list,text_translated
0,1. Blueprints for 3D-printed firearms should b...,Blueprints for 3D-printed firearms should be p...,1.,Topic,17593,3D Printer and Guns: Should Blueprints of 3D-P...,3d-printer-and-guns-should-blueprints-of-3d-pr...,Topic,0,Blueprints for 3D-printed firearms should be p...,"[0.0065077078, 0.0308416, 0.009240101, 0.03242...",en,"gun,government,science",Blueprints for 3D-printed firearms should be p...
1,1.1. Pro: 3D Printing Makes Illegal Proliferat...,3D Printing Makes Illegal Proliferation of Gun...,1.1.,Pro,17593,3D Printer and Guns: Should Blueprints of 3D-P...,3d-printer-and-guns-should-blueprints-of-3d-pr...,Pro,1,3D Printing Makes Illegal Proliferation of Gun...,"[0.0054018325, 0.044243313, 0.002241337, -0.00...",en,"gun,government,science",3D Printing Makes Illegal Proliferation of Gun...
2,1.1.1. Con: It is [notoriously](https://money....,It is [notoriously](https://money.cnn.com/2015...,1.1.1.,Con,17593,3D Printer and Guns: Should Blueprints of 3D-P...,3d-printer-and-guns-should-blueprints-of-3d-pr...,Con,2,It is notoriously easy to buy guns in America....,"[-0.00086407, 0.052978553, -0.007538742, -0.00...",en,"gun,government,science",It is [notoriously](https://money.cnn.com/2015...
3,1.1.2. Pro: With 3D Printers even indidividual...,With 3D Printers even indidividuals who wouldn...,1.1.2.,Pro,17593,3D Printer and Guns: Should Blueprints of 3D-P...,3d-printer-and-guns-should-blueprints-of-3d-pr...,Pro,2,With 3D Printers even indidividuals who wouldn...,"[-0.019681733, 0.10858277, 0.00873058, 0.02045...",en,"gun,government,science",With 3D Printers even indidividuals who wouldn...
4,1.2. Pro: The opposition to 3D guns is based o...,The opposition to 3D guns is based on the exis...,1.2.,Pro,17593,3D Printer and Guns: Should Blueprints of 3D-P...,3d-printer-and-guns-should-blueprints-of-3d-pr...,Pro,1,The opposition to 3D guns is based on the exis...,"[0.046481628, 0.02652415, 0.011560266, 0.02954...",en,"gun,government,science",The opposition to 3D guns is based on the exis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
348538,1.4.3. Con: Negative consequences of acknowled...,Negative consequences of acknowledging the tru...,1.4.3.,Con,6695,Does One Vote Make A Difference?,does-one-vote-make-a-difference-6695.txt,Pro,2,Negative consequences of acknowledging the tru...,"[-0.01478132, 0.03886117, 0.007119894, -0.0061...",en,"elections,society,truth,politics",Negative consequences of acknowledging the tru...
348539,1.4.3.1. Con: It does if the factors in play r...,It does if the factors in play requires the mi...,1.4.3.1.,Con,6695,Does One Vote Make A Difference?,does-one-vote-make-a-difference-6695.txt,Con,3,It does if the factors in play requires the mi...,"[-0.04228716, 0.048748165, -0.0048393654, -0.0...",en,"elections,society,truth,politics",It does if the factors in play requires the mi...
348540,1.5. Con: At the margin one vote makes a diffe...,At the margin one vote makes a difference.,1.5.,Con,6695,Does One Vote Make A Difference?,does-one-vote-make-a-difference-6695.txt,Con,1,At the margin one vote makes a difference.,"[-0.035363145, 0.027356112, -0.0057576136, -0....",en,"elections,society,truth,politics",At the margin one vote makes a difference.
348541,1.5.1. Pro: In small constituencies an individ...,In small constituencies an individual vote mat...,1.5.1.,Pro,6695,Does One Vote Make A Difference?,does-one-vote-make-a-difference-6695.txt,Con,2,In small constituencies an individual vote mat...,"[-0.03212115, 0.0392083, -0.01595761, 0.005190...",en,"elections,society,truth,politics",In small constituencies an individual vote mat...


time: 29.7 s (started: 2023-04-26 09:54:30 +00:00)


In [3]:
data[['text','all-mpnet-base-v2']]

,text,all-mpnet-base-v2
0,Blueprints for 3D-printed firearms should be p...,"[0.0065077078, 0.0308416, 0.009240101, 0.03242..."
1,3D Printing Makes Illegal Proliferation of Gun...,"[0.0054018325, 0.044243313, 0.002241337, -0.00..."
2,It is [notoriously](https://money.cnn.com/2015...,"[-0.00086407, 0.052978553, -0.007538742, -0.00..."
3,With 3D Printers even indidividuals who wouldn...,"[-0.019681733, 0.10858277, 0.00873058, 0.02045..."
4,The opposition to 3D guns is based on the exis...,"[0.046481628, 0.02652415, 0.011560266, 0.02954..."
...,...,...
348538,Negative consequences of acknowledging the tru...,"[-0.01478132, 0.03886117, 0.007119894, -0.0061..."
348539,It does if the factors in play requires the mi...,"[-0.04228716, 0.048748165, -0.0048393654, -0.0..."
348540,At the margin one vote makes a difference.,"[-0.035363145, 0.027356112, -0.0057576136, -0...."
348541,In small constituencies an individual vote mat...,"[-0.03212115, 0.0392083, -0.01595761, 0.005190..."


time: 191 ms (started: 2023-04-26 10:06:39 +00:00)


In [4]:
# generate embeddings for each model
sentence = 'abortion'
embeddings_x = np.asarray(embed(model_st1, 'sentence transformer', [sentence]))[0]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

time: 5.48 s (started: 2023-04-26 10:09:20 +00:00)


In [6]:
len(embeddings_x)

768

time: 2.97 ms (started: 2023-04-26 10:10:05 +00:00)


In [7]:
from scipy import spatial


time: 447 µs (started: 2023-04-26 10:10:11 +00:00)


In [8]:
data['cosine_similarity'] = data['all-mpnet-base-v2'].progress_apply(lambda x: 1 - spatial.distance.cosine(x, embeddings_x))

  0%|          | 0/348543 [00:00<?, ?it/s]

time: 28.2 s (started: 2023-04-26 10:10:29 +00:00)


In [12]:
data[['text_translated','all-mpnet-base-v2']][data['cosine_similarity']>0.5]['text_translated'][2240]

,text_translated,all-mpnet-base-v2
2240,Legislation governing abortions could seriousl...,"[-0.01604087, 0.07311805, 0.012503733, 0.01953..."
2731,Not opting to terminate a pregnancy is still a...,"[-0.005186898, 0.03327702, 0.024078427, -0.079..."
7329,Cannabis harms the unborn child.,"[-0.07292552, 0.08028247, 0.0006957138, -0.049..."
13467,Thesis: Women have the right to abortion freely.,"[0.0053827716, 0.11796835, 0.019734368, -0.036..."
13472,For: The right to abortion is connected to the...,"[-0.0003012311, 0.04767374, 0.0028510836, -0.0..."
...,...,...
344346,It suggests people who want to have an abortio...,"[0.030107446, 0.044736356, -0.0048137866, -0.0..."
344616,[People have a right](https://en.wikipedia.org...,"[-0.0038188414, 0.06865455, 0.011977701, -0.03..."
344645,[People have a right](https://en.wikipedia.org...,"[-0.0038188219, 0.068654545, 0.011977705, -0.0..."
345444,"A clear, middle stance like ""A woman's third a...","[0.042649835, 0.027444357, 0.020575056, -0.019..."


time: 57.1 ms (started: 2023-04-26 10:13:00 +00:00)


In [13]:
data[['text_translated','all-mpnet-base-v2']][data['cosine_similarity']>0.5]['text_translated'][2240]

'Legislation governing abortions could seriously endanger lives https://www.npr.org/sections/health-shots/2019/05/02/688260025/new-trump-rule-protects-health-care-workers-who-refuse-care-for-religious-reason as it means health care workers could deny critical - and even life saving - treatment. This goes against a basic Christian principle of preserving life as well as contravening the constitutional right to life https://www.law.cornell.edu/constitution/amendmentxiv#targetText=Section%201.&targetText=No%20state%20shall%20make%20or,equal%20protection%20of%20the%20laws..'

time: 85.4 ms (started: 2023-04-26 10:17:20 +00:00)
